Build Gold — Live Production Data
Reads silver battery data, builds weekly and monthly aggregated features, saves to gold.

**Input**: silver/erp/battery/battery_clean_live.json
**Output**: gold/erp/battery/phase1_overall_weekly_live.parquet, phase1_overall_monthly_live.parquet

In [0]:
%run ./_local_config

In [0]:
import sys
import pandas as pd
sys.path.append("/Workspace/Users/venura-it@brownsgroup.com/Exide sales/Exide-Sales-Forecast")

from src.io.storage import get_blob_service, read_silver, save_gold
from src.transform.build_gold_features import build_gold_overall_weekly, build_gold_overall_monthly

blob_service = get_blob_service(storage_account_name, storage_account_key)

In [0]:
silver = read_silver(blob_service, "live/battery_clean_live.json")
silver["posting_date"] = pd.to_datetime(silver["posting_date"])

print(f"Silver: {silver.shape}")
print(f"Date range: {silver['posting_date'].min()} to {silver['posting_date'].max()}")

Sales prediction

In [0]:
# Weekly gold — safe to use all data
gold_weekly = build_gold_overall_weekly(silver)

# Monthly gold — trim to the last COMPLETE month first
monthly_cutoff = silver["posting_date"].max().replace(day=1) - pd.Timedelta(days=1)
silver_for_monthly = silver[silver["posting_date"] <= monthly_cutoff].copy()
gold_monthly = build_gold_overall_monthly(silver_for_monthly)

print(f"Weekly gold: {gold_weekly.shape}")
print(gold_weekly.tail(5)[["week_start", "total_units_sold"]])

print(f"\nMonthly gold (cutoff: {monthly_cutoff.date()}): {gold_monthly.shape}")
print(gold_monthly.tail(5)[["month_start", "total_units_sold"]])

In [0]:
print(f"Filled weeks: {gold_weekly['was_filled'].sum()} / {len(gold_weekly)}")
print(f"Filled months: {gold_monthly['was_filled'].sum()} / {len(gold_monthly)}")

In [0]:
save_gold(blob_service, gold_weekly, "live/phase1_overall_weekly_live.parquet")
save_gold(blob_service, gold_monthly, "live/phase1_overall_monthly_live.parquet")
print("Saved weekly and monthly gold")

Brand wise prediction

In [0]:
from src.transform.build_gold_features import (
    build_gold_brand_weekly, build_gold_brand_monthly
)

gold_brand_weekly = build_gold_brand_weekly(silver)

monthly_cutoff = silver["posting_date"].max().replace(day=1) - pd.Timedelta(days=1)
silver_for_monthly = silver[silver["posting_date"] <= monthly_cutoff].copy()
gold_brand_monthly = build_gold_brand_monthly(silver_for_monthly)

print(f"Brand weekly: {gold_brand_weekly.shape}")
print(gold_brand_weekly.groupby("brand_code")["total_units_sold"].sum())

print(f"\nBrand monthly: {gold_brand_monthly.shape}")
print(gold_brand_monthly.groupby("brand_code")["total_units_sold"].sum())

In [0]:
save_gold(blob_service, gold_brand_weekly, "live/phase2_brand_weekly_live.parquet")
save_gold(blob_service, gold_brand_monthly, "live/phase2_brand_monthly_live.parquet")
print("Saved brand weekly and monthly gold")